In [21]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Detect platform                               ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

WORK_DIR  = '/kaggle/working' if ON_KAGGLE else '/content'
REPO_DIR  = f'{WORK_DIR}/repo'
CKPT_DIR  = f'{WORK_DIR}/checkpoints'

os.makedirs(CKPT_DIR, exist_ok=True)

print(f"Platform : {PLATFORM}")
print(f"Work dir : {WORK_DIR}")
print(f"Repo dir : {REPO_DIR}")
print(f"Ckpt dir : {CKPT_DIR}")

Platform : kaggle
Work dir : /kaggle/working
Repo dir : /kaggle/working/repo
Ckpt dir : /kaggle/working/checkpoints


In [23]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load secrets (GitHub token + rclone config)   ║
# ╚══════════════════════════════════════════════════════════╝
#
# FIRST TIME SETUP — do this on each platform once:
#
# On Kaggle:
#   Notebook → Add-ons → Secrets → Add new secret
#   Add two secrets:
#     Name: GITHUB_TOKEN   Value: (your GitHub personal access token)
#     Name: RCLONE_CONF    Value: (contents of ~/.config/rclone/rclone.conf on your PC)
#
# On Colab:
#   Left sidebar → key icon → Secrets → Add new secret
#   Same two secrets: GITHUB_TOKEN and RCLONE_CONF
#
# To get your GitHub token:
#   github.com → Settings → Developer settings → Personal access tokens → Generate new token
#   Give it "repo" scope (full control of repositories)
#
# To get your rclone config:
#   Run on your PC:  cat ~/.config/rclone/rclone.conf
#   Copy the entire output as the value of RCLONE_CONF

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets       = UserSecretsClient()
    GITHUB_TOKEN  = secrets.get_secret("GITHUB_TOKEN")
    RCLONE_CONF   = secrets.get_secret("RCLONE_CONF")
elif ON_COLAB:
    from google.colab import userdata
    GITHUB_TOKEN  = userdata.get('GITHUB_TOKEN')
    RCLONE_CONF   = userdata.get('RCLONE_CONF')

# Write rclone config to the expected location
import pathlib
rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
rclone_cfg.write_text(RCLONE_CONF)

# Set your GitHub details here (change these to yours)
GITHUB_USER = "rayedriasat"
GITHUB_REPO = "cse465-voice-project"
NOTEBOOK_FILENAME = "cse465-approach2.ipynb"   # exact filename in your repo

print("Secrets loaded.")
print(f"GitHub repo : {GITHUB_USER}/{GITHUB_REPO}")

Secrets loaded.
GitHub repo : rayedriasat/cse465-voice-project


In [24]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Install rclone                                ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess

result = subprocess.run(
    'curl https://rclone.org/install.sh | sudo bash',
    shell=True, capture_output=True, text=True
)
# Verify
ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
print(ver.stdout.split('\n')[0])   # prints e.g. "rclone v1.67.0"

rclone v1.73.3


In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Clone / pull the repo (gets your notebook)    ║
# ╚══════════════════════════════════════════════════════════╝
#
# This pulls the latest version of your notebook from GitHub.
# If the repo already exists this session, it just does git pull.

GITHUB_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(f'git clone {GITHUB_URL} {REPO_DIR}', shell=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(f'git -C {REPO_DIR} pull', shell=True)

# Configure git identity (needed for commits)
subprocess.run(f'git -C {REPO_DIR} config user.email "rayedriasat@gmail.com"', shell=True)
subprocess.run(f'git -C {REPO_DIR} config user.name "rayedriasat"', shell=True)

# List repo contents
print("\nRepo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"  {f}")

Loading processor...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/1846 [00:00<?, ?it/s]

SeamlessM4Tv2ForSpeechToSpeech LOAD REPORT from: facebook/seamless-m4t-v2-large
Key                                                      | Status     |  | 
---------------------------------------------------------+------------+--+-
text_encoder.layers.{0...23}.self_attn.out_proj.bias     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn.fc2.weight              | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.k_proj.bias       | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.v_proj.weight     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.out_proj.weight   | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn.fc1.weight              | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.k_proj.weight     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.v_proj.bias       | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.q_proj.weight     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn.fc1.bias                | UNEXPECTED |  | 
text_enc

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded. Parameters: 1805.513349 M


In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Pull checkpoints from Drive                   ║
# ╚══════════════════════════════════════════════════════════╝
#
# Downloads whatever checkpoints are saved on your Drive.
# First session: nothing to download, that's fine.

print("Pulling checkpoints from Google Drive...")
result = subprocess.run(
    f'rclone sync gdrive:cse465/checkpoints {CKPT_DIR}',
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    print("Warning:", result.stderr[:300])

ckpt_files = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')]
if ckpt_files:
    print(f"Downloaded {len(ckpt_files)} checkpoint(s):")
    for f in sorted(ckpt_files):
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f"  {f:<50} {mb:.1f} MB")
else:
    print("No checkpoints yet (fresh start).")

Loading one LibriSpeech example...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Original text : CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
Duration      : 3.5s

▶ Original English audio:



Running inference (English → Bengali)...
Output shape  : (45760,)
Output sample rate: 16000

▶ Bengali output audio:



Both files saved to /kaggle/working/checkpoints/


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Define save/load/commit helper functions      ║
# ╚══════════════════════════════════════════════════════════╝

import torch, glob, json
from datetime import datetime

def save_checkpoint(state: dict, name: str, step: int, keep: int = 3):
    """
    Call this during training every N steps.
    Saves locally then immediately pushes to Drive.
    """
    filename  = f"{name}_step{step:06d}.pt"
    local_path = f"{CKPT_DIR}/{filename}"

    torch.save(state, local_path)
    print(f"[ckpt] Saved: {filename}")

    # Push to Drive right away — don't wait for end of session
    r = subprocess.run(
        f'rclone copy {local_path} gdrive:cse465/checkpoints/',
        shell=True, capture_output=True, text=True
    )
    print(f"[ckpt] Drive push: {'OK' if r.returncode == 0 else 'FAILED — ' + r.stderr[:100]}")

    # Clean up old local copies, keep only last `keep`
    old_files = sorted(glob.glob(f"{CKPT_DIR}/{name}_step*.pt"))
    for old in old_files[:-keep]:
        os.remove(old)
        print(f"[ckpt] Removed old local: {os.path.basename(old)}")


def load_latest_checkpoint(name: str):
    """
    Call this at the start of a training cell.
    Looks locally first (already pulled in Cell 5), returns None if nothing found.
    """
    files = sorted(glob.glob(f"{CKPT_DIR}/{name}_step*.pt"))
    if not files:
        print(f"[ckpt] No checkpoint for '{name}' — starting fresh.")
        return None
    latest = files[-1]
    state  = torch.load(latest, map_location='cpu')
    print(f"[ckpt] Loaded: {os.path.basename(latest)}  (step {state.get('step','?')})")
    return state


def commit_and_push(message: str = None):
    """
    Saves the current notebook back to GitHub.
    Call this whenever you make meaningful changes to the notebook.
    """
    if message is None:
        message = f"auto-save {datetime.now().strftime('%Y-%m-%d %H:%M')}"

    nb_src  = f"{WORK_DIR}/{NOTEBOOK_FILENAME}"   # where Kaggle/Colab keeps the live notebook
    nb_dest = f"{REPO_DIR}/{NOTEBOOK_FILENAME}"

    # Copy the live notebook file into the repo folder
    if os.path.exists(nb_src):
        subprocess.run(f'cp "{nb_src}" "{nb_dest}"', shell=True)
    else:
        # On Kaggle the notebook path is slightly different — find it
        result = subprocess.run(
            f'find {WORK_DIR}/.. -name "{NOTEBOOK_FILENAME}" 2>/dev/null | head -1',
            shell=True, capture_output=True, text=True
        )
        found = result.stdout.strip()
        if found:
            subprocess.run(f'cp "{found}" "{nb_dest}"', shell=True)
        else:
            print(f"[git] Could not find {NOTEBOOK_FILENAME} — skipping notebook copy")

    subprocess.run(f'git -C {REPO_DIR} add -A', shell=True)

    # Check if there's anything to commit
    status = subprocess.run(
        f'git -C {REPO_DIR} status --porcelain',
        shell=True, capture_output=True, text=True
    )
    if not status.stdout.strip():
        print("[git] Nothing changed — no commit needed.")
        return

    subprocess.run(f'git -C {REPO_DIR} commit -m "{message}"', shell=True)
    r = subprocess.run(
        f'git -C {REPO_DIR} push {GITHUB_URL} main',
        shell=True, capture_output=True, text=True
    )
    print(f"[git] Push: {'OK' if r.returncode == 0 else 'FAILED — ' + r.stderr[:200]}")


def session_status():
    """Quick summary of what's saved where."""
    print("=" * 55)
    print(f"  Platform  : {PLATFORM}")
    print(f"  Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

    local = sorted(glob.glob(f"{CKPT_DIR}/*.pt"))
    print(f"\n  Local checkpoints ({len(local)}):")
    for f in local:
        print(f"    {os.path.basename(f):<45} {os.path.getsize(f)/1e6:.1f} MB")

    drive = subprocess.run(
        'rclone lsf gdrive:cse465/checkpoints/',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    drive_files = [l for l in drive.split('\n') if l.endswith('.pt')]
    print(f"\n  Drive checkpoints ({len(drive_files)}):")
    for f in drive_files:
        print(f"    {f}")
    print("=" * 55)


print("Helper functions ready.")
print("  save_checkpoint(state, name, step)")
print("  load_latest_checkpoint(name)")
print("  commit_and_push(message)")
print("  session_status()")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Install ML packages                           ║
# ╚══════════════════════════════════════════════════════════╝

subprocess.run([
    'pip', 'install', '-q',
    'transformers',
    'datasets',
    'torchaudio',
    'speechbrain',
    'peft',
    'librosa',
    'jiwer',
    'evaluate',
], check=True)

print("Packages installed.")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  LAST CELL — Run before closing every session           ║
# ╚══════════════════════════════════════════════════════════╝

# Final Drive sync — catches anything not pushed mid-session
print("Final sync to Drive...")
subprocess.run(
    f'rclone sync {CKPT_DIR} gdrive:cse465/checkpoints/',
    shell=True
)

# Save a log file
log = {
    'platform'  : PLATFORM,
    'time'      : datetime.now().isoformat(),
    'completed' : 'EDIT THIS — what you finished this session',
    'next'      : 'EDIT THIS — what to do next session',
    'ckpts'     : [os.path.basename(f) for f in glob.glob(f'{CKPT_DIR}/*.pt')],
}
log_path = f"{CKPT_DIR}/session_log.json"
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)
subprocess.run(f'rclone copy {log_path} gdrive:cse465/checkpoints/', shell=True)
print(json.dumps(log, indent=2))

# Push notebook + any code changes to GitHub
commit_and_push("session end: " + log['completed'])

session_status()